In [1]:
import pandas as pd
import sqlite3
import os
import smtplib
import sys
import time
from datetime import datetime
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.utils import formataddr

# ✅ パス設定共通
try:
    base_dir = os.path.dirname(__file__)
except NameError:
    base_dir = os.getcwd()

sys.path.append(os.path.abspath(os.path.join(base_dir, "..")))
from utils.config2 import simple_conditions, PROJECT_DIR, GSHEET_NAME, SHEET_NAME

start_time = time.time()

# ✅ DBパス
user_base = os.path.join(
    os.environ["USERPROFILE"] if os.name == 'nt' else os.path.expanduser("~"),
    "myenv310", PROJECT_DIR
)
db_path = os.path.join(user_base, "db", "output.db")
print(f"[INFO] 使用DB: {db_path}")

# ✅ メールに表示する基本カラム（DBに無い可能性があるので後で存在チェック）
display_columns = ["台番号", "機種名", "BIG", "REG", "ATART", "最終ゲーム", "前日最終ゲーム数", "初回当選ゲーム数", "宵越し累計ゲーム数", "最大放出数", "すろらぼURL", "pscubeURL", "宵越し特賞履歴ステータス1回前", "宵越し特賞履歴ゲーム1回前", "宵越し特賞履歴ステータス2回前", "宵越し特賞履歴ゲーム2回前", "宵越し特賞履歴ステータス3回前", "宵越し特賞履歴ゲーム3回前"]

# ✅ 条件式から必要カラム
needed_from_conditions = (
    [c for a in simple_conditions for c in a[:2]] 
)
base_needed = ["実行日", "台番号", "すろらぼURL", "pscubeURL", "取得更新日", "機種名", "BIG", "REG", "ATART", "累計ゲーム", "最大放出数", "最終ゲーム", "前日最終ゲーム数", "宵越し累計ゲーム数", "宵越し特賞履歴ステータス1回前", "宵越し特賞履歴ゲーム1回前", "宵越し特賞履歴ステータス2回前", "宵越し特賞履歴ゲーム2回前", "宵越し特賞履歴ステータス3回前", "宵越し特賞履歴ゲーム3回前"]
needed_cols = list(dict.fromkeys(needed_from_conditions + base_needed))

# ✅ DBに存在する列だけをSELECT（安全）
with sqlite3.connect(db_path) as conn:
    pragma = pd.read_sql_query("PRAGMA table_info(result_table);", conn)
    existing_cols = set(pragma["name"].tolist())
    select_cols = [c for c in needed_cols if c in existing_cols]
    if not select_cols:
        raise RuntimeError("result_table から取得できる列が見つかりません。テーブル名やスキーマを確認してください。")
    sql = f'''SELECT {", ".join([f"[{col}]" for col in select_cols])} FROM result_table ORDER BY ROWID DESC'''
    df = pd.read_sql_query(sql, conn)

# ✅ 日付処理と最新行抽出
df["実行日"] = pd.to_datetime(df["実行日"], errors='coerce')
df = df.dropna(subset=["実行日"])
max_date = df["実行日"].dt.date.max()
df_latest = (
    df[df["実行日"].dt.date == max_date]
    .sort_values("実行日", ascending=False)
    .drop_duplicates(subset=["台番号"])
).copy()

print(f"[INFO] 最新日: {max_date} 件数: {len(df_latest)}")

# ---- URL列→アンカー（列名に'URL'を含む全列）
def convert_url_columns_to_anchor(df_in: pd.DataFrame) -> pd.DataFrame:
    df_out = df_in.copy()
    url_cols = [c for c in df_out.columns if "URL" in str(c)]
    def to_anchor(v):
        if isinstance(v, str):
            s = v.strip()
            if s.startswith("http://") or s.startswith("https://"):
                return f'<a href="{s}" target="_blank">リンク</a>'
        return v
    for col in url_cols:
        df_out[col] = df_out[col].apply(to_anchor)
    return df_out

# ======================
# メール用リンク列を追加（DBには保存しない）
# ======================
base_url = f"https://sedoinfinity.xsrv.jp/{PROJECT_DIR}/machines/machine_"
df_latest["サイト"] = df_latest["台番号"].astype(str).apply(
    lambda x: f'<a href="{base_url}{x}.html" target="_blank">リンク</a>'
)

# ★ ここで URL列を一括アンカー化（すろらぼURL / pscubeURL / img_url_a 等）
df_latest = convert_url_columns_to_anchor(df_latest)

# ✅ SMTP設定
smtp_server = "smtp.gmail.com"
port = 587
sender_email = "straydog12341234@gmail.com"
password = "cyam jmtc pgfr slpx"
receiver_emails = ["straydog12341234@gmail.com", "enaena@shchiba.uk"]

def send_email(subject, html_body):
    msg = MIMEMultipart("alternative")
    msg["From"] = formataddr((GSHEET_NAME, sender_email))
    msg["To"] = ", ".join(receiver_emails)
    msg["Subject"] = subject
    msg.attach(MIMEText(html_body, "html"))
    try:
        with smtplib.SMTP(smtp_server, port) as server:
            server.starttls()
            server.login(sender_email, password)
            server.sendmail(sender_email, receiver_emails, msg.as_string())
        print(f"✅ メール送信完了: {subject}")
    except Exception as e:
        print(f"[ERROR] メール送信失敗: {e}")

# ✅ 表組み生成の共通関数（存在する列だけを使う）
def build_mail_table(df_src, extra_cols):
    # 表示基本列（DBにないものはスキップ。ただし「リンク」はdf_latestで作成済み）
    base_cols = [c for c in display_columns if (c == "サイト" or c in df_src.columns)]
    # 追加列（条件列など）も存在チェック
    extra_cols = [c for c in extra_cols if c in df_src.columns]
    cols = list(dict.fromkeys(base_cols + extra_cols))
    df_mail = df_src.sort_values("台番号", ascending=True)[cols]

    # 小数を強制的に整数化（NaNは0にしてからintに変換）
    for c in df_mail.select_dtypes(include="number").columns:
        df_mail[c] = df_mail[c].fillna(0).astype(int)

    table_html = df_mail.to_html(index=False, escape=False, border=1, classes="styled-table")
    return table_html


# ---- 共通CSS（最小追加）
TABLE_CSS = """
<style>
.styled-table{border-collapse:collapse;width:100%}
.styled-table th,.styled-table td{border:1px solid #ccc;padding:5px;text-align:left}
.styled-table th{background-color:#f2f2f2}
.section{margin:14px 0 22px}
.section h2{margin:0 0 8px}
hr{border:none;border-top:1px solid #ddd;margin:16px 0}
.small{color:#666;font-size:12px}
</style>
"""

# ★★★★★ 集約メール用セクションバッファ（ここが主変更点）★★★★★
all_sections = []
total_hits = 0

# ✅ 単純条件チェック（送信せず、セクションHTMLを貯める）
for col, border_col, label in simple_conditions:
    df_temp = df_latest.copy()
    df_temp[col] = pd.to_numeric(df_temp.get(col), errors='coerce')
    df_temp[border_col] = pd.to_numeric(df_temp.get(border_col), errors='coerce')

    df_filtered = df_temp[
        df_temp[col].notna() & df_temp[border_col].notna() & (df_temp[col] > df_temp[border_col])
    ].copy()

    if df_filtered.empty:
        print(f"[INFO] スキップ: {label} 一致なし")
        continue

    table_html = build_mail_table(df_filtered, [col, border_col])
    section_html = f"""
    <div class="section">
      <h2>{label} 条件一致 {len(df_filtered)}件 ({max_date})</h2>
      {table_html}
    </div>
    """
    all_sections.append(section_html)
    total_hits += len(df_filtered)


# ✅ まとめて1通送信（セクションがあるときのみ）
if all_sections:
    header = f"<h1>{GSHEET_NAME}{SHEET_NAME} 集約通知</h1><div class='small'>対象日: {max_date} / 合計ヒット: {total_hits}</div><hr>"
    full_html = f"<html><head>{TABLE_CSS}</head><body>{header}{''.join(all_sections)}</body></html>"
    subject = f"{GSHEET_NAME}{SHEET_NAME} 集約通知 {max_date}（{total_hits}件）"
    send_email(subject, full_html)
else:
    print("[INFO] 条件一致なしのためメール送信しません。")

end_time = time.time()
print(f"[INFO] スクリプト完了（実行時間: {end_time - start_time:.2f} 秒）")


[INFO] 使用DB: /home/ubuntu/myenv310/ootake-maruhachi-s/db/output.db
[INFO] 最新日: 2026-05-08 件数: 167
[INFO] スキップ: 前日最終ゲーム数と初回当選ゲーム数の合計 一致なし
✅ メール送信完了: 大竹マルハチslot 集約通知 2026-05-08（9件）
[INFO] スクリプト完了（実行時間: 12.41 秒）
